# Workflow Interface 102 - Held out aggregator validation
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/intel/openfl/blob/develop/openfl-tutorials/experimental/workflow/102_Aggregator_Validation.ipynb)

In this tutorial, we build on the ideas from the [first](https://github.com/securefederatedai/openfl/blob/develop/openfl-tutorials/experimental/workflow/101_MNIST.ipynb) quick start notebook, and demonstrate how to perform validation on the aggregator after training.

# Getting Started

First we start by installing the necessary dependencies for the workflow interface

In [1]:
!pip install git+https://github.com/securefederatedai/openfl.git
!pip install -r workflow_interface_requirements.txt
!pip install torch
!pip install torchvision

# Uncomment this if running in Google Colab
#!pip install -r https://raw.githubusercontent.com/intel/openfl/develop/openfl-tutorials/experimental/workflow/workflow_interface_requirements.txt
#import os
#os.environ["USERNAME"] = "colab"

  Cloning https://github.com/securefederatedai/openfl.git to /tmp/pip-req-build-ferae3vm
  Running command git clone --filter=blob:none --quiet https://github.com/securefederatedai/openfl.git /tmp/pip-req-build-ferae3vm


  Resolved https://github.com/securefederatedai/openfl.git to commit cd71616c3c09592e0e7193ad0fa60ea8f4cf54a4


  Installing build dependencies ... -

 \

 |

...

 done



[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip



[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip



[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip



[notice] A new release of pip is available: 25.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


We begin with the quintessential example of a small pytorch CNN model trained on the MNIST dataset. Let's start define our dataloaders, model, optimizer, and some helper functions like we would for any other deep learning experiment

In [2]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch
import torchvision
import numpy as np

n_epochs = 3
batch_size_train = 64
batch_size_test = 1000
learning_rate = 0.01
momentum = 0.5
log_interval = 10

random_seed = 1
torch.backends.cudnn.enabled = False
torch.manual_seed(random_seed)

mnist_train = torchvision.datasets.MNIST('files/', train=True, download=True,
                             transform=torchvision.transforms.Compose([
                               torchvision.transforms.ToTensor(),
                               torchvision.transforms.Normalize(
                                 (0.1307,), (0.3081,))
                             ]))

mnist_test = torchvision.datasets.MNIST('files/', train=False, download=True,
                             transform=torchvision.transforms.Compose([
                               torchvision.transforms.ToTensor(),
                               torchvision.transforms.Normalize(
                                 (0.1307,), (0.3081,))
                             ]))

class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.conv2_drop = nn.Dropout2d()
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2_drop(self.conv2(x)), 2))
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, training=self.training)
        x = self.fc2(x)
        return F.log_softmax(x)
    
def inference(network,test_loader):
    network.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
      for data, target in test_loader:
        output = network(data)
        test_loss += F.nll_loss(output, target, size_average=False).item()
        pred = output.data.max(1, keepdim=True)[1]
        correct += pred.eq(target.data.view_as(pred)).sum()
    test_loss /= len(test_loader.dataset)
    print('\nTest set: Avg. loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
      test_loss, correct, len(test_loader.dataset),
      100. * correct / len(test_loader.dataset)))
    accuracy = float(correct / len(test_loader.dataset))
    return accuracy

Failed to download (trying next):
HTTP Error 404: Not Found



  0%|                                                                                                          | 0/9912422 [00:00<?, ?it/s]

  1%|▌                                                                                         | 65536/9912422 [00:00<00:18, 535670.88it/s]

  4%|███▏                                                                                    | 360448/9912422 [00:00<00:05, 1619000.42it/s]

 14%|████████████▎                                                                          | 1409024/9912422 [00:00<00:01, 4742695.68it/s]

...

100%|███████████████████████████████████████████████████████████████████████████████████████| 1648877/1648877 [00:00<00:00, 4538960.39it/s]

Extracting files/MNIST/raw/t10k-images-idx3-ubyte.gz to files/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



  0%|                                                                                                             | 0/4542 [00:00<?, ?it/s]

100%|█████████████████████████████████████████████████████████████████████████████████████████████| 4542/4542 [00:00<00:00, 3161913.49it/s]

Extracting files/MNIST/raw/t10k-labels-idx1-ubyte.gz to files/MNIST/raw



Next we import the `FLSpec`, `LocalRuntime`, and placement decorators.

- `FLSpec` – Defines the flow specification. User defined flows are subclasses of this.
- `Runtime` – Defines where the flow runs, infrastructure for task transitions (how information gets sent). The `LocalRuntime` runs the flow on a single node.
- `aggregator/collaborator` - placement decorators that define where the task will be assigned

In [3]:
from copy import deepcopy

from openfl.experimental.workflow.interface import FLSpec, Aggregator, Collaborator
from openfl.experimental.workflow.runtime import LocalRuntime
from openfl.experimental.workflow.placement import aggregator, collaborator


def FedAvg(models, weights=None):
    new_model = models[0]
    state_dicts = [model.state_dict() for model in models]
    state_dict = new_model.state_dict()
    for key in models[1].state_dict():
        state_dict[key] = torch.from_numpy(np.average([state[key].numpy() for state in state_dicts],
                                                      axis=0, 
                                                      weights=weights))
    new_model.load_state_dict(state_dict)
    return new_model


Now we come to the updated flow definition. Here we use the same tasks as the [quickstart](https://github.com/securefederatedai/openfl/blob/develop/openfl-tutorials/experimental/workflow/101_MNIST.ipynb), but give the aggregator a `test_loader` as a private attribute. The aggregator will do a forward pass on each of the aggregator's models using it's validation data, and weight the highest accuracy model higher than others.  

In [4]:
class AggregatorValidationFlow(FLSpec):

    def __init__(self, model = None, optimizer = None, rounds=3, **kwargs):
        super().__init__(**kwargs)
        if model is not None:
            self.model = model
            self.optimizer = optimizer
        else:
            self.model = Net()
            self.optimizer = optim.SGD(self.model.parameters(), lr=learning_rate,
                                   momentum=momentum)
        self.rounds = rounds

    @aggregator
    def start(self):
        print(f'Performing initialization for model')
        self.collaborators = self.runtime.collaborators
        self.private = 10
        self.current_round = 0
        self.next(self.aggregated_model_validation,foreach='collaborators',exclude=['private'])

    @collaborator
    def aggregated_model_validation(self):
        print(f'Performing aggregated model validation for collaborator {self.input}')
        self.agg_validation_score = inference(self.model,self.test_loader)
        print(f'{self.input} value of {self.agg_validation_score}')
        self.next(self.train)

    @collaborator
    def train(self):
        self.model.train()
        self.optimizer = optim.SGD(self.model.parameters(), lr=learning_rate,
                                   momentum=momentum)
        train_losses = []
        for batch_idx, (data, target) in enumerate(self.train_loader):
          self.optimizer.zero_grad()
          output = self.model(data)
          loss = F.nll_loss(output, target)
          loss.backward()
          self.optimizer.step()
          if batch_idx % log_interval == 0:
            print('Train Epoch: 1 [{}/{} ({:.0f}%)]\tLoss: {:.6f}'.format(
               batch_idx * len(data), len(self.train_loader.dataset),
              100. * batch_idx / len(self.train_loader), loss.item()))
            self.loss = loss.item()
            torch.save(self.model.state_dict(), 'model.pth')
            torch.save(self.optimizer.state_dict(), 'optimizer.pth')
        self.training_completed = True
        self.next(self.local_model_validation)

    @collaborator
    def local_model_validation(self):
        self.local_validation_score = inference(self.model,self.test_loader)
        print(f'Doing local model validation for collaborator {self.input}: {self.local_validation_score}')
        self.next(self.join, exclude=['training_completed'])

    @aggregator
    def join(self,inputs):
        self.average_loss = sum(input.loss for input in inputs)/len(inputs)
        self.aggregated_model_accuracy = sum(input.agg_validation_score for input in inputs)/len(inputs)
        self.local_model_accuracy = sum(input.local_validation_score for input in inputs)/len(inputs)
        print(f'Average aggregated model validation values = {self.aggregated_model_accuracy}')
        print(f'Average training loss = {self.average_loss}')
        print(f'Average local model validation values = {self.local_model_accuracy}')
        
        highest_accuracy = 0
        highest_accuracy_model_idx = -1
        for idx,col in enumerate(inputs):
            accuracy_for_held_out_agg_data = inference(col.model,self.test_loader)
            if accuracy_for_held_out_agg_data > highest_accuracy:
                highest_accuracy = accuracy_for_held_out_agg_data
                highest_accuracy_model_idx = idx
        
        relative_model_weights = len(inputs)*[1]
        # Give highest accuracy model (on held out aggregator data) 2x the importance
        relative_model_weights[highest_accuracy_model_idx] = 2
        print(f'Aggregator validation score: {highest_accuracy}')
        print(f'Highest accuracy model sent from {inputs[highest_accuracy_model_idx].input}. Receiving 2x weight in updated model')
        self.model = FedAvg([input.model for input in inputs],weights=relative_model_weights)
        self.optimizer = [input.optimizer for input in inputs][0]
        self.current_round += 1
        if self.current_round < self.rounds:
            self.next(self.aggregated_model_validation, foreach='collaborators', exclude=['private'])
        else:
            self.next(self.end)
        
    @aggregator
    def end(self):
        print(f'This is the end of the flow')  

Aggregator step "start" registered
Collaborator step "aggregated_model_validation" registered
Collaborator step "train" registered
Collaborator step "local_model_validation" registered
Aggregator step "join" registered
Aggregator step "end" registered


You'll notice in the `FederatedFlow` definition above that there were certain attributes that the flow was not initialized with, namely the `train_loader` and `test_loader` for each of the collaborators. Each participant has it's own set of private attributes which can be set using callback function while instantiating the participant. The callback function returns the private attributes (`train_loader` & `test_loader`) in form of a dictionary where the key is the attribute name, and the value is the object that will be made accessible to that participant's task

Below, we segment shards of the MNIST dataset for **four collaborators**: `Portland`, `Seattle`, `Chandler`, and `Portland`. Each has their own slice of the dataset that is accessible through the `train_loader` and `test_loader` attributes, which are set using the `callable_to_initialize_collaborator_private_attributes` callable function. Note that the private attributes are flexible, and you can choose to pass in a completely different type of object to any of the collaborators or aggregator (with an arbitrary name). These private attributes will always be filtered out of the current state when transfering from collaborator to aggregator, or vice versa.

Private attributes can be set using callback function while instantiating the participant. Parameters required by the callback function are specified as arguments while instantiating the participant. In this example callback function, `callable_to_initialize_collaborator_private_attributes`, returns the private attributes `train_loader` and `test_loader` of the collaborator. Callback function, `callable_to_initialize_aggregator_private_attributes`, returns the private attribute `test_loader` of the Aggregator.

In [5]:
collaborator_names = ['Portland', 'Seattle', 'Chandler', 'Bangalore']

def callable_to_initialize_aggregator_private_attributes(n_collaborators, test_dataset, batch_size_train):
    aggregator_test = deepcopy(test_dataset)
    aggregator_test.targets = test_dataset.targets[n_collaborators::n_collaborators+1]
    aggregator_test.data = test_dataset.data[n_collaborators::n_collaborators+1]
    return {
        'test_loader': torch.utils.data.DataLoader(aggregator_test,batch_size=batch_size_train, shuffle=True)
    }

# Setup Aggregator private attributes via callable function
aggregator = Aggregator(
    name="agg",
    private_attributes_callable=callable_to_initialize_aggregator_private_attributes,
    n_collaborators=len(collaborator_names),
    test_dataset=mnist_test, batch_size_train=batch_size_train
)

# Setup collaborators private attributes via callable function
def callable_to_initialize_collaborator_private_attributes(index, n_collaborators, train_dataset, test_dataset, batch_size_train):
    local_train = deepcopy(train_dataset)
    local_test = deepcopy(test_dataset)
    local_train.data = train_dataset.data[index::n_collaborators]
    local_train.targets = train_dataset.targets[index::n_collaborators]
    local_test.data = test_dataset.data[index::n_collaborators]
    local_test.targets = test_dataset.targets[index::n_collaborators]
    
    return {
        'train_loader': torch.utils.data.DataLoader(local_train,batch_size=batch_size_train, shuffle=True),
        'test_loader': torch.utils.data.DataLoader(local_test,batch_size=batch_size_train, shuffle=True)
    }

collaborators=[]
for idx, collaborator_name in enumerate(collaborator_names):
    collaborators.append(
        Collaborator(
            name=collaborator_name, num_cpus=0, num_gpus=0,
            private_attributes_callable=callable_to_initialize_collaborator_private_attributes,
            index=idx, n_collaborators=len(collaborator_names),
            train_dataset=mnist_train, test_dataset=mnist_test, batch_size_train=batch_size_train,
        )
    )

local_runtime = LocalRuntime(aggregator=aggregator, collaborators=collaborators, backend='ray')
print(f'Local runtime collaborators = {local_runtime.collaborators}')

2025-02-12 04:30:44,045	INFO worker.py:1724 -- Started a local Ray instance.


No GPUs found! If this is a mistake please try running "nvidia-smi --list-gpus" manually.


creating actor with 0, 0


Local runtime collaborators = ['Portland', 'Seattle', 'Chandler', 'Bangalore']


Now that we have our flow and runtime defined, let's run the experiment! 

In [6]:
model = None
best_model = None
optimizer = None
flflow = AggregatorValidationFlow(model, optimizer)
flflow.runtime = local_runtime
flflow.run()

Creating local datastore in current directory (/var/github/workspace/openfl/payalcha_openfl/openfl-tutorials/experimental/workflow/.metaflow)


# Congratulations!
Now that you've completed your this notebook, see some of the more advanced things you can do in our [other tutorials](https://github.com/securefederatedai/openfl/tree/develop/openfl-tutorials/experimental/workflow), including:

- Using the LocalRuntime Ray Backend for dedicated GPU access
- Vertical Federated Learning
- Model Watermarking
- Differential Privacy
- And More!